In [9]:
import sympy as sp
import numpy as np
from IPython.display import display, Math
import warnings
warnings.filterwarnings("ignore")

sp.init_printing(use_latex='mathjax')
t, f = sp.symbols('t f', real=True)
pi = sp.pi
j_sym = sp.Symbol('j') # Símbolo seguro para la j

def destructor_graficas_examen(tramos):
    print("==================================================")
    print(" 🎯 DESTRUCTOR DE GRÁFICAS: EXAMEN MA2N (V3.0)")
    print("==================================================\n")
    
    w = 2 * pi * f 
    
    # ----------------------------------------------------
    # INCISO 1: PLANTEAR LA INTEGRAL
    # ----------------------------------------------------
    print("1. PLANTEAMIENTO DE LA INTEGRAL (Cópialo en tu hoja):")
    integral_planteada = 0
    for funcion_t, t_min, t_max in tramos:
        integral_planteada += sp.Integral(funcion_t * sp.exp(-j_sym * 2 * pi * f * t), (t, t_min, t_max))
    
    display(Math(rf"X(f) = {sp.latex(integral_planteada)}"))
    
    # ----------------------------------------------------
    # INCISO 2: CÁLCULO DE LA TRANSFORMADA
    # ----------------------------------------------------
    print("\n2. TRANSFORMADA CALCULADA X(f):")
    X_f_cruda = 0
    for funcion_t, t_min, t_max in tramos:
        res = sp.integrate(funcion_t * sp.exp(-sp.I * w * t), (t, t_min, t_max))
        if isinstance(res, sp.Piecewise):
            res = res.args[0][0]
        X_f_cruda += res
    
    X_f_limpia = sp.simplify(sp.expand(X_f_cruda).rewrite(sp.sin))
    
    # Cambio seguro de 'I' a 'j' a nivel de variable simbólica, no de texto
    X_f_visual = X_f_limpia.subs(sp.I, j_sym)
    display(Math(rf"X(f) = {sp.latex(X_f_visual)}"))
    
    print("\n💡 TIP DEL INGENIERO PARA EL INCISO 2:")
    print("-> Si la respuesta tiene 'sin(A*f) / f', es un pulso rectangular: usa 'sinc(f)'.")
    print("-> Si la respuesta tiene 'sin^2(A*f) / f^2', vino de un triángulo: usa 'sinc^2(f)'.")
    print("-> Si solo hay 'cos(A*f)', son Impulsos de Dirac en el tiempo.")
    
    # ----------------------------------------------------
    # INCISO 3A: VALOR MÁXIMO
    # ----------------------------------------------------
    print("\n3A. VALOR MÁXIMO |X(f)| (Evaluando límite f->0):")
    try:
        max_val = sp.limit(X_f_limpia, f, 0)
        display(Math(rf"|X(f)|_{{max}} = {sp.latex(max_val)}"))
    except:
        print("La función diverge o no tiene máximo en f=0.")

    # ----------------------------------------------------
    # INCISO 3B: PRIMER CERO
    # ----------------------------------------------------
    print("\n3B. PRIMER CERO DE LA FUNCIÓN (f > 0):")
    try:
        X_func = sp.lambdify(f, X_f_limpia, 'numpy')
        f_vals = np.linspace(0.001, 10.0, 10000) 
        y_vals = np.real(X_func(f_vals))
        cruces = np.where(np.diff(np.signbit(y_vals)))[0]
        
        if len(cruces) > 0:
            primer_cero = f_vals[cruces[0]]
            display(Math(rf"f \approx {primer_cero:.4f} \text{{ Hz}}"))
            display(Math(rf"\text{{(O lo que es igual: }} \omega \approx {primer_cero * 2 * np.pi:.4f} \text{{ rad/s)}}"))
        else:
            print("No se detectó ningún cruce por cero en las primeras frecuencias.")
    except Exception as e:
        print(f"No se pudo escanear el cero: {e}")
        
    print("\n==================================================")

# ==========================================
# 📝 ZONA DE CONFIGURACIÓN
# ==========================================
# (funcion_t, limite_inferior, limite_superior)

mis_tramos = [
    (5 * sp.DiracDelta(t + 3), -sp.oo, sp.oo),  # La flecha de la izquierda (en -3)
    (5 * sp.DiracDelta(t - 3), -sp.oo, sp.oo)   # La flecha de la derecha (en 3)
]

destructor_graficas_examen(mis_tramos)

 🎯 DESTRUCTOR DE GRÁFICAS: EXAMEN MA2N (V3.0)

1. PLANTEAMIENTO DE LA INTEGRAL (Cópialo en tu hoja):


<IPython.core.display.Math object>


2. TRANSFORMADA CALCULADA X(f):


<IPython.core.display.Math object>


💡 TIP DEL INGENIERO PARA EL INCISO 2:
-> Si la respuesta tiene 'sin(A*f) / f', es un pulso rectangular: usa 'sinc(f)'.
-> Si la respuesta tiene 'sin^2(A*f) / f^2', vino de un triángulo: usa 'sinc^2(f)'.
-> Si solo hay 'cos(A*f)', son Impulsos de Dirac en el tiempo.

3A. VALOR MÁXIMO |X(f)| (Evaluando límite f->0):


<IPython.core.display.Math object>


3B. PRIMER CERO DE LA FUNCIÓN (f > 0):


<IPython.core.display.Math object>

<IPython.core.display.Math object>

#  GUÍA DE SUPERVIVENCIA MA2N - TERCER PARCIAL

Esta guía conecta los ejercicios que vendrán en tu examen con el script exacto que debes usar para resolverlos. 

---

##  INVENTARIO DEL ARSENAL (Tus Scripts)

1. **`asistenteTeoremas.ipynb` (La Caja Negra):** Resuelve transformadas directas e inversas al instante (ideal para comprobar respuestas).
2. **`PasosInversa.ipynb` (El Deconstructor):** Genera el procedimiento de fracciones parciales escrito paso a paso para el examen.
3. **`Graficador_Examen.ipynb` (El Pintor):** Dibuja $f(t)$ y $|F(\omega)|$ al instante.
4. **`Destructor_Graficas.ipynb` (El Francotirador):** Si te viene un dibujo de un pulso, te saca la integral, la función sinc, el valor máximo y el primer cero.
5. **`definiciondetallada.ipynb` (El Músculo):** Solo se usa si el ingeniero te obliga a integrar "por definición". Te desglosa toda el álgebra.

---

### PROTOCOLO DE EXAMEN (Paso a Paso por Temario)

### PUNTO 1: Transformada inversa por formulario y gráficas
**Cómo identificarlo:** Te dan un $F(\omega)$ (probablemente una fracción) y te piden hallar $f(t)$ y hacer los dibujos.

**Flujo de trabajo:**
1. **Saca la respuesta:** Abre `asistenteTeoremas.ipynb`. Mete la ecuación en `mi_F_frecuencia` y ejecuta. Copia la $f(t)$ que te dé.
   * *Ejemplo:* `mi_F_frecuencia = 1 / ((1 + j*w) * (1 + 2*j*w))`
2. **Haz las gráficas:** Abre `Graficador_Examen.ipynb`. Mete tu $F(\omega)$ y la $f(t)$ que acabas de encontrar. Dale Play y copia los dos dibujos a tu hoja.

### PUNTO 2: Inversa por fracciones parciales
**Cómo identificarlo:** Va pegado al punto 1, te piden justificar la inversa usando fracciones.

**Flujo de trabajo:**
1. **Genera el texto:** Abre `PasosInversa.ipynb`. Mete tu $F(\omega)$ en `F_examen` y tu $f(t)$ en `f_caja_negra`. 
2. **Copia a la hoja:** El script te imprimirá la fracción separada algebraicamente (ej. $\frac{2}{...} - \frac{1}{...}$). Copia esa separación y los pasos escritos para asegurar tus puntos de procedimiento.

### PUNTO 3: Resolución de Ecuación Diferencial
**Cómo identificarlo:** Te dan un circuito o una ecuación que tiene derivadas, como $i' + i = v(t)$.

**Flujo de trabajo (Este es manual asistido):**
1. **Aplica Fourier a la ecuación (a mano):** Si tienes $i' + 3i = v(t)$, recuerda que la derivada $i'$ se vuelve $j\omega I(\omega)$. Te quedará: $j\omega I(\omega) + 3I(\omega) = V(\omega)$.
2. **Despeja la salida:** Factoriza $I(\omega)$: $I(\omega)(j\omega + 3) = V(\omega) \implies I(\omega) = V(\omega) \cdot \frac{1}{j\omega + 3}$
3. **Mete la Transformada del Voltaje:** Si el voltaje era $v(t) = e^{-2t}u(t)$, su transformada es $\frac{1}{j\omega + 2}$. Multiplicas ambas fracciones.
   * Tu ecuación a resolver ahora es: $I(\omega) = \frac{1}{(j\omega + 2)(j\omega + 3)}$
4. **Usa tus Scripts:** ¡Acabas de convertir el problema 3 en el problema 2! Mete esa fracción final a tu `PasosInversa.ipynb` para que te haga las fracciones parciales y te dé la corriente final $i(t)$ en el tiempo.

### PUNTO SORPRESA: Análisis de Gráficas (Sinc, Ceros y Máximos)
**Cómo identificarlo:** Te ponen un dibujo de cuadrados (pulsos) o flechas (impulsos) y te piden $X(f)$, el primer cero o el valor máximo.

**Flujo de trabajo:**
1. **Traduce el dibujo:** Abre `Destructor_Graficas.ipynb`. Observa la altura del bloque y dónde empieza/termina.
2. **Configura:** Mete los datos en `mis_tramos`. 
   * *Ejemplo (bloque de altura 2, de -1 a 1):* `mis_tramos = [(2, -1, 1)]`
3. **Copia todo:** El script te dará:
   * La integral planteada para el inciso 1.
   * La transformada para pasar a "sinc" para el inciso 2.
   * El valor máximo y el cruce por cero para el inciso 3.

---

## ALERTAS DE SINTAXIS EN SYMPY (Kryptonitas a evitar)

* **Multiplicación:** NUNCA escribas `jw` o `2t`. Siempre pon el asterisco: `j*w` o `2*t`.
* **La 'j' imaginaria:** Asegúrate de estar usando la variable `j` que configuramos en los scripts (que equivale a `sp.I`), y no la letra 'i' normal de tu teclado.
* **Exponenciales:** Se escriben `sp.exp()`. Ejemplo: $e^{-3t}$ es `sp.exp(-3*t)`.
* **Potencias:** Se usa doble asterisco `**`. Ejemplo: $\omega^2$ es `w**2`.
* **Trampa de fracciones parciales:** El comando `sp.apart` odia las exponenciales ($e$). Si tu fracción tiene una $e^{-j3\omega}$ arriba, sácala de la ecuación temporalmente en tu mente, corre el script de fracciones parciales solo con el denominador, y a la respuesta final $f(t)$ simplemente réstale un 3 a todas las $t$ por teorema de corrimiento.